In [1]:
# import duckdb

# # Convert via a single SQL query
# duckdb.query("COPY (SELECT * FROM 'credit_card_default_ssb_dataset_50000.csv') TO 'credit_card_default_ssb_dataset_50000.parquet' (FORMAT PARQUET)")


In [2]:
from ssb_experimental import *
from utils.universal_data_loader import *

# Example Feature Groupings Setup
business_groups = {
"delinquency_vars":["dpd_avg_3m", "dpd_avg_6m", "dpd_avg_12m","max_dpd_12m"],            
"transaction_vars":["txn_count_avg_3m", "txn_count_avg_6m", "txn_count_avg_12m"],
"spend_vars":["spend_avg_3m", "spend_avg_6m", "spend_avg_12m"],
"repayment_vars":["payment_ratio_avg_3m","payment_ratio_avg_6m","payment_ratio_avg_12m"],
"card_utilization_vars":["utilization_avg_3m","utilization_avg_6m","utilization_avg_12m","utilization_max_12m"],
}

# 2. Define your dynamically scaled optimization grid
my_search_grid = {
    "min_sample_size": [10000, 5000,500,100],
    "min_lift": [2.0,1.5]
}


# Initialize the builder
builder = StrategicSegmentBuilder(
    target="default_flag",
    n_jobs=-1,
    min_sample_size=5000,
    min_lift=1.5,
    min_events=100,
    top_n_vars=30,
    max_segments=10,
    enable_diversity=True,
    max_feature_reuse=1,
    enable_1way=True,  # Include 1-way rules from final output
    enable_2way=True,  # Exclude 2-way rules from final output
    enable_3way=True,  # Exclude 3-way rules from final output
    feature_groups=None,
    ignore_features=None,
    param_grid=my_search_grid
)


data = UniversalDataLoader(file_path=r"/workspaces/Strategic_Segment_Builder/Datasets/all_categorical_credit_default_50000.csv").load()

segments_df = builder.extract_segments(data)
final_eval = builder.evaluate_final_coverage(data)

2026-07-09 14:37:58,110 | INFO     | [universal_data_loader.py:74] | Detecting local file format handler for extension: '.csv'...
2026-07-09 14:37:58,172 | INFO     | [ssb_experimental.py:337] | DuckDB Configured: Threads=4/4, MemoryLimit=7GB
2026-07-09 14:37:58,293 | INFO     | [ssb_experimental.py:352] | Dynamic Grid Search Enabled: 8 total configurations.
2026-07-09 14:37:58,296 | INFO     | [ssb_experimental.py:373] | Iteration 1 | Remaining Volume: 50,000 | Base Rate: 14.77%
2026-07-09 14:38:00,521 | INFO     | [ssb_experimental.py:509] | Feature Usage Tracker Update -> 'payment_band' used count = 1
2026-07-09 14:38:00,522 | INFO     | [ssb_experimental.py:509] | Feature Usage Tracker Update -> 'delinquency_band' used count = 1
2026-07-09 14:38:00,522 | INFO     | [ssb_experimental.py:524] | Segment 1 Captured (Size Floor: 100 | Lift Floor: 2.0): payment_band IN ('Missed') AND delinquency_band IN ('60Plus')
2026-07-09 14:38:00,526 | INFO     | [ssb_experimental.py:373] | Iteration

In [8]:
from prettytable import PrettyTable
import pandas as pd
table = PrettyTable()
table.field_names = list(pd.DataFrame(final_eval).columns)
for _, row in pd.DataFrame(final_eval).iterrows():
    table.add_row(list(row))
print(table)

+---------+-------------+---------------+--------------------+--------------------+--------------------+--------------------+
| segment | total_count | target_events |   response_rate    | base_response_rate |    capture_rate    |        lift        |
+---------+-------------+---------------+--------------------+--------------------+--------------------+--------------------+
|   0.0   |   42081.0   |     5100.0    | 12.119483852570044 |       14.768       |       84.162       | 0.8206584407211568 |
|   1.0   |    376.0    |     279.0     | 74.20212765957447  |       14.768       |       0.752        | 5.024521103708997  |
|   2.0   |    7543.0   |     2005.0    | 26.580935967121835 |       14.768       | 15.085999999999999 | 1.7999008645125836 |
+---------+-------------+---------------+--------------------+--------------------+--------------------+--------------------+


In [9]:
print("--- FULL SEGMENT RULES ---\n")

for index, row in pd.DataFrame(segments_df).iterrows():
    print(f"Segment ID: {row['segment_id']}")
    print(f"Raw Rule:   {row['rule_string']}")
    print(f"SQL Filter: {row['sql_filter']}")
    print("-" * 50)

--- FULL SEGMENT RULES ---

Segment ID: 1
Raw Rule:   payment_band=<ArrowStringArray>
['Missed']
Length: 1, dtype: str & delinquency_band=<ArrowStringArray>
['60Plus']
Length: 1, dtype: str
SQL Filter: payment_band IN ('Missed') AND delinquency_band IN ('60Plus')
--------------------------------------------------
Segment ID: 2
Raw Rule:   utilization_band=<ArrowStringArray>
['VeryHigh']
Length: 1, dtype: str
SQL Filter: utilization_band IN ('VeryHigh')
--------------------------------------------------


In [11]:
builder.explain_feature_journey("delinquency_band")

AUDIT TRAIL FOR FEATURE: 'delinquency_band'

[Iteration 1]
  • Current dynamic IV   : 36.7684
  • Previous times used  : 0
  • Selection Status     : Eligible for Combination Search
  🎉 SELECTED as part of winning rule!
     Rule: payment_band=<ArrowStringArray>
['Missed']
Length: 1, dtype: str & delinquency_band=<ArrowStringArray>
['60Plus']
Length: 1, dtype: str

[Iteration 2]
  • Current dynamic IV   : 30.1628
  • Previous times used  : 1
  • Selection Status     : Excluded (Max Feature Reuse Exceeded)
  • Winner this round    : utilization_band=<ArrowStringArray>
['VeryHigh']
Length: 1, dtype: str (Variables: ['utilization_band'])

[Iteration 3]
  • Current dynamic IV   : 33.0017
  • Previous times used  : 1
  • Selection Status     : Excluded (Max Feature Reuse Exceeded)
